In [3]:
!pip install imbalanced-learn


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline  # <— important!
import joblib
import os

# -------------------------------
#  Load and clean dataset
# -------------------------------
data = pd.read_csv(r"D:\WildFire Project\data\CA_Weather_Fire_Dataset_1984-2025.csv")

rename_map = {
    "MAX_TEMP": "temperature",
    "PRECIPITATION": "rainfall",
    "AVG_WIND_SPEED": "wind_speed",
    "SEASON": "season",
    "FIRE_START_DAY": "fire_risk"
}
data = data.rename(columns=rename_map)
data = data.dropna(subset=["temperature", "rainfall", "wind_speed", "season", "fire_risk"])
data["fire_risk"] = data["fire_risk"].astype(int)

#  Features and Target
X = data[["temperature", "rainfall", "wind_speed", "season"]]
y = data["fire_risk"]

#  Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#  Preprocessing
numeric_features = ["temperature", "rainfall", "wind_speed"]
categorical_features = ["season"]

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

#  SMOTE
smote = SMOTE(random_state=42)

#  Model
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight=None,  # Let SMOTE handle balancing, no need for class_weight here
    max_depth=12
)

#  Build an Imbalanced-Learn Pipeline
model = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("classifier", rf)
])

#  Train the model
model.fit(X_train, y_train)

#  Save model
joblib.dump(model, "wildfire_risk_model.pkl")
print("✅ Model trained and saved successfully as wildfire_risk_model.pkl")
print("Model path:", os.path.abspath("wildfire_risk_model.pkl"))

#  Model Evaluation
y_pred = model.predict(X_test)

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Accuracy Score
print("Accuracy Score:", accuracy_score(y_test, y_pred))

✅ Model trained and saved successfully as wildfire_risk_model.pkl
Model path: d:\WildFire Project\notebooks\wildfire_risk_model.pkl

Confusion Matrix:
[[1513  489]
 [ 300  694]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.76      0.79      2002
           1       0.59      0.70      0.64       994

    accuracy                           0.74      2996
   macro avg       0.71      0.73      0.72      2996
weighted avg       0.75      0.74      0.74      2996

Accuracy Score: 0.736648865153538
